# Data Preprocessing & Feature Engineering

## Objectives
- Load and validate the dataset.
- Separate features from the target variable.
- Assess feature suitability.
- Prepare data for machine learning.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/AIML Dataset.csv")

In [3]:
df.shape

(6362620, 11)

In [4]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='str')

In [5]:
X = df[["type","amount","oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest",]]

y = df["isFraud"]

In [6]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

X shape: (6362620, 6)
y shape: (6362620,)

Feature columns:
['type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

Target distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
  X, y, train_size=0.8, test_size=0.2, random_state=42, stratify=y
)

In [8]:
x_train.shape,x_test.shape,y_train.shape,y_test.shape

((5090096, 6), (1272524, 6), (5090096,), (1272524,))

In [9]:
print("Training set:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting set:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True) * 100)

Training set:
isFraud
0    5083526
1       6570
Name: count, dtype: int64
isFraud
0    99.870926
1     0.129074
Name: proportion, dtype: float64

Testing set:
isFraud
0    1270881
1       1643
Name: count, dtype: int64
isFraud
0    99.870887
1     0.129113
Name: proportion, dtype: float64


In [10]:
print(x_train.dtypes)

print("\nTransaction types:")
print(x_train["type"].value_counts())

type                  str
amount            float64
oldbalanceOrg     float64
newbalanceOrig    float64
oldbalanceDest    float64
newbalanceDest    float64
dtype: object

Transaction types:
type
CASH_OUT    1790307
PAYMENT     1721234
CASH_IN     1119026
TRANSFER     426382
DEBIT         33147
Name: count, dtype: int64


In [11]:
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.compose import ColumnTransformer

categorical_features = ["type"]
numeric_features = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

In [12]:

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
                dtype=np.float32
            ),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numeric_features
        )
    ],
    remainder="drop",
    sparse_threshold=1.0
)

In [13]:
print(preprocessor)

ColumnTransformer(sparse_threshold=1.0,
                  transformers=[('categorical',
                                 OneHotEncoder(dtype=<class 'numpy.float32'>,
                                               handle_unknown='ignore'),
                                 ['type']),
                                ('numerical', StandardScaler(),
                                 ['amount', 'oldbalanceOrg', 'newbalanceOrig',
                                  'oldbalanceDest', 'newbalanceDest'])])


In [14]:
print("Training shape:", x_train.shape)
print("Testing shape:", x_test.shape)

print(
    "Training memory usage:",
    round(x_train.memory_usage(deep=True).sum() / (1024 ** 2), 2),
    "MB"
)

print(
    "Testing memory usage:",
    round(x_test.memory_usage(deep=True).sum() / (1024 ** 2), 2),
    "MB"
)

Training shape: (5090096, 6)
Testing shape: (1272524, 6)
Training memory usage: 506.9 MB
Testing memory usage: 126.72 MB


In [15]:
X_train_transformed = preprocessor.fit_transform(x_train)

print("Shape:", X_train_transformed.shape)
print("Data type:", type(X_train_transformed))
print("Data type of values:", X_train_transformed.dtype)

Shape: (5090096, 10)
Data type: <class 'scipy.sparse._csr.csr_matrix'>
Data type of values: float64


In [16]:
train_memory_mb = (
    X_train_transformed.data.nbytes
    + X_train_transformed.indices.nbytes
    + X_train_transformed.indptr.nbytes
) / (1024 ** 2)

print("Transformed training memory:", round(train_memory_mb, 2), "MB")
print("Non-zero values:", X_train_transformed.nnz)
print("Total possible values:", X_train_transformed.shape[0] * X_train_transformed.shape[1])

Transformed training memory: 368.93 MB
Non-zero values: 30540576
Total possible values: 50900960


In [17]:
X_test_transformed = preprocessor.transform(x_test)

print("Test shape:", X_test_transformed.shape)
print("Data type:", type(X_test_transformed))
print("Data type of values:", X_test_transformed.dtype)

Test shape: (1272524, 10)
Data type: <class 'scipy.sparse._csr.csr_matrix'>
Data type of values: float64


In [18]:
print("Training features:", X_train_transformed.shape)
print("Testing features:", X_test_transformed.shape)

print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

print("Training fraud cases:", y_train.sum())
print("Testing fraud cases:", y_test.sum())

Training features: (5090096, 10)
Testing features: (1272524, 10)
Training labels: (5090096,)
Testing labels: (1272524,)
Training fraud cases: 6570
Testing fraud cases: 1643


In [19]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

dummy_model = DummyClassifier(strategy="most_frequent")

dummy_model.fit(X_train_transformed, y_train)

y_pred_dummy = dummy_model.predict(X_test_transformed)

print("Accuracy:", accuracy_score(y_test, y_pred_dummy))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dummy, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dummy))

Accuracy: 0.9987088652159016

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.00      0.00      0.00      1643

    accuracy                           1.00   1272524
   macro avg       0.50      0.50      0.50   1272524
weighted avg       1.00      1.00      1.00   1272524


Confusion Matrix:
[[1270881       0]
 [   1643       0]]


## 3. Model Training — Logistic Regression

In [20]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    class_weight="balanced",
    solver="saga",
    max_iter=100,
    random_state=42
)

In [21]:
import time

start_time = time.time()

logistic_model.fit(X_train_transformed,y_train)

end_time = time.time()

print("Training completed.")
print("Training time:", round(end_time - start_time, 2), "seconds")

Training completed.
Training time: 432.9 seconds


c:\Users\Afkar\OneDrive\Desktop\fraud-detection\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [22]:
y_pred_logistic = logistic_model.predict(X_test_transformed)

print("Predictions completed.")
print("Number of predictions:", len(y_pred_logistic))

Predictions completed.
Number of predictions: 1272524


In [23]:
print("Classification Report:")
print(
    classification_report(
        y_test,
        y_pred_logistic,
        target_names=["Legitimate", "Fraud"],
        zero_division=0
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_logistic))

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      0.95      0.97   1270881
       Fraud       0.02      0.93      0.04      1643

    accuracy                           0.95   1272524
   macro avg       0.51      0.94      0.51   1272524
weighted avg       1.00      0.95      0.97   1272524

Confusion Matrix:
[[1201944   68937]
 [    120    1523]]


In [24]:
x_train_new, x_val, y_train_new, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print("Training:", x_train_new.shape)
print("Validation:", x_val.shape)
print("Training fraud cases:", y_train_new.sum())
print("Validation fraud cases:", y_val.sum())

Training: (4072076, 6)
Validation: (1018020, 6)
Training fraud cases: 5256
Validation fraud cases: 1314


In [25]:

X_train_new_transformed = preprocessor.fit_transform(x_train_new)

X_val_transformed = preprocessor.transform(x_val)

X_test_transformed = preprocessor.transform(x_test)

print("Training:", X_train_new_transformed.shape)
print("Validation:", X_val_transformed.shape)
print("Testing:", X_test_transformed.shape)

Training: (4072076, 10)
Validation: (1018020, 10)
Testing: (1272524, 10)


In [26]:
logistic_model = LogisticRegression(
    class_weight="balanced",
    solver="saga",
    max_iter=200,
    random_state=42,
    C=1
)

start_time = time.time()

logistic_model.fit(
    X_train_new_transformed,
    y_train_new
)

print("Training time:", round(time.time() - start_time, 2), "seconds")
print("Iterations:", logistic_model.n_iter_)

Training time: 534.57 seconds
Iterations: [200]


c:\Users\Afkar\OneDrive\Desktop\fraud-detection\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [27]:

y_val_proba = logistic_model.predict_proba(
    X_val_transformed
)[:, 1]

print("Minimum probability:", y_val_proba.min())
print("Maximum probability:", y_val_proba.max())

Minimum probability: 1.0501962286990704e-299
Maximum probability: 1.0


In [30]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

thresholds = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]

results = []

for threshold in thresholds:
    y_val_pred = (y_val_proba >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, y_val_pred, zero_division=0
        ),
        "Recall": recall_score(y_val, y_val_pred),
        "F1-score": f1_score(y_val, y_val_pred),
        "Predicted Fraud": int(y_val_pred.sum())
    })

results_df = pd.DataFrame(results)

print(results_df.round(4).to_string(index=False))

print(
    "\nValidation Average Precision:",
    round(average_precision_score(y_val, y_val_proba), 4)
)

 Threshold  Precision  Recall  F1-score  Predicted Fraud
      0.10     0.0041  0.9992    0.0083           316744
      0.30     0.0090  0.9954    0.0179           145167
      0.50     0.0214  0.9117    0.0419            55905
      0.70     0.0457  0.8303    0.0867            23865
      0.90     0.1471  0.6933    0.2428             6191
      0.95     0.2402  0.6362    0.3487             3481
      0.99     0.4925  0.5472    0.5184             1460

Validation Average Precision: 0.5557


In [ ]:
selected_threshold = 0.99

y_test_proba = logistic_model.predict_proba(
    X_test_transformed
)[:, 1]

y_test_pred = (
    y_test_proba >= selected_threshold
).astype(int)

print("Selected Threshold:", selected_threshold)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=["Legitimate", "Fraud"],
        zero_division=0
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print(
    "\nTest Average Precision:",
    round(average_precision_score(y_test, y_test_proba), 4)
)

Selected Threshold: 0.99

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00   1270881
       Fraud       0.49      0.55      0.52      1643

    accuracy                           1.00   1272524
   macro avg       0.75      0.77      0.76   1272524
weighted avg       1.00      1.00      1.00   1272524

Confusion Matrix:
[[1269965     916]
 [    746     897]]

Test Average Precision: 0.5606
